### ⚙️ Inicialización de la Sesión de Spark

Crearemos la sesión con un driver específico para la lectura de la base de datos. Para establecer la conexión, introduciremos:
* `URL`
* `Usuario` y `Contraseña`
* `Driver`

💡 **Nota de rendimiento:** Esto nos permite crear los DataFrames sin cargar los datos directamente en la RAM, difiriendo la ejecución hasta que se realice una acción.

In [1]:
import os
import findspark
findspark.init()

from pyspark.sql import SparkSession

# --- LA LÍNEA MÁGICA PARA WINDOWS ---
os.environ["HADOOP_HOME"] = "D:\\hadoop"
# ------------------------------------

# 1. SETUP CON EL DRIVER DE POSTGRESQL
# Añadimos la configuración 'spark.jars.packages' apuntando a la versión estable del driver.
spark = SparkSession.builder \
    .appName("M5_JDBC_Connection_Full") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .config("spark.driver.memory", "24g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.driver.maxResultSize", "8g") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "4g") \
    .master("local[*]") \
    .getOrCreate()

print("Sesión iniciada. 24GB de RAM asignados y driver JDBC descargado/listo...")

# 2. CONFIGURACIÓN DE LA CONEXIÓN
# Asumiendo que tu BD está en localhost y el puerto por defecto es 5432
db_url = "jdbc:postgresql://localhost:5432/m5_db"

# IMPORTANTE: Cambia 'tu_usuario' y 'tu_contraseña' por los que usas para entrar a pgAdmin
db_properties = {
    "user": "postgres",         # Usuario por defecto, cámbialo si es distinto
    "password": "9794", 
    "driver": "org.postgresql.Driver"
}

# 3. EXTRACCIÓN DE DATOS (Lectura distribuida)
print("Conectando a PostgreSQL y leyendo tablas...")

# Spark lee los metadatos y crea los DataFrames, pero no carga toda la data 
# a la memoria RAM hasta que haces una acción (como .show() o .count())
df_calendar = spark.read.jdbc(url=db_url, table="stg_calendar", properties=db_properties)
df_prices = spark.read.jdbc(url=db_url, table="stg_prices", properties=db_properties)
df_sales = spark.read.jdbc(url=db_url, table="stg_sales", properties=db_properties)

# 4. VERIFICACIÓN
print("¡Tablas cargadas exitosamente como DataFrames de PySpark!")
print("Muestra de stg_sales:")
df_sales.show(5)

d:\Usuarios\Escritorio\M5\m5-forecasting-project\venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Sesión iniciada. 24GB de RAM asignados y driver JDBC descargado/listo...
Conectando a PostgreSQL y leyendo tablas...
¡Tablas cargadas exitosamente como DataFrames de PySpark!
Muestra de stg_sales:
+--------------------+-------------+---------+-------+--------+--------+-----+---------+
|                  id|      item_id|  dept_id| cat_id|store_id|state_id|    d|sales_qty|
+--------------------+-------------+---------+-------+--------+--------+-----+---------+
|HOBBIES_1_001_CA_...|HOBBIES_1_001|HOBBIES_1|HOBBIES|    CA_1|      CA|d_229|        0|
|HOBBIES_1_002_CA_...|HOBBIES_1_002|HOBBIES_1|HOBBIES|    CA_1|      CA|d_229|        0|
|HOBBIES_1_003_CA_...|HOBBIES_1_003|HOBBIES_1|HOBBIES|    CA_1|      CA|d_229|        0|
|HOBBIES_1_004_CA_...|HOBBIES_1_004|HOBBIES_1|HOBBIES|    CA_1|      CA|d_229|        0|
|HOBBIES_1_005_CA_...|HOBBIES_1_005|HOBBIES_1|HOBBIES|    CA_1|      CA|d_229|        3|
+--------------------+-------------+---------+-------+--------+--------+-----+---------+
on

### 🔄 Asignación de Tipos de Datos (Casting)

Ahora ajustaremos el esquema de nuestro DataFrame asignando los tipos correspondientes a cada columna:
* **Ventas** ➔ `Integer`
* **Fecha** ➔ `DateType`
* **Precios** ➔ `Float`

💡 **Verificación:** Al finalizar estas conversiones, comprobaremos el esquema para asegurarnos de que Spark ha recibido y aplicado esta información correctamente.

In [2]:
from pyspark.sql.types import IntegerType, DateType, StringType, FloatType

print("Aplicando esquemas estrictos para optimizar memoria...")

# 1. Tipado de stg_sales (La tabla más pesada)
# Casteamos las ventas a Integer para reducir la RAM a la mitad (de 64 a 32 bits).
df_sales_typed = df_sales \
    .withColumn("sales_qty", df_sales["sales_qty"].cast(IntegerType())) \
    .withColumn("d", df_sales["d"].cast(StringType()))

# 2. Tipado de stg_calendar
# Fundamental para Time Series: Asegurar que 'date' es DateType real.
df_calendar_typed = df_calendar \
    .withColumn("date", df_calendar["date"].cast(DateType())) \
    .withColumn("wm_yr_wk", df_calendar["wm_yr_wk"].cast(IntegerType()))

# 3. Tipado de stg_prices
# Los precios los manejamos como Float (32 bits) en lugar de Double (64 bits)
df_prices_typed = df_prices \
    .withColumn("sell_price", df_prices["sell_price"].cast(FloatType())) \
    .withColumn("wm_yr_wk", df_prices["wm_yr_wk"].cast(IntegerType()))

print("¡Esquemas optimizados exitosamente!")

# Verificamos que Spark ha entendido los nuevos tipos
df_sales_typed.printSchema()

Aplicando esquemas estrictos para optimizar memoria...
¡Esquemas optimizados exitosamente!
root
 |-- id: string (nullable = true)
 |-- item_id: string (nullable = true)
 |-- dept_id: string (nullable = true)
 |-- cat_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- state_id: string (nullable = true)
 |-- d: string (nullable = true)
 |-- sales_qty: integer (nullable = true)



### 🧹 Limpieza de Datos: Nulos y Atípicos

Pasamos a la fase de calidad de datos, donde nos centraremos en analizar y corregir nuestras variables numéricas clave. Nos enfocaremos en dos tareas principales:
* **Valores Nulos:** Identificación y gestión de datos faltantes en el dataset.
* **Valores Atípicos (Outliers):** Detección y tratamiento de registros con valores extremos o anómalos.
* **Columnas objetivo:** `Precios` y `Ventas`.

💡 **Nota analítica:** Tratar correctamente estos valores es un paso crítico para no alterar las distribuciones reales de las ventas y evitar sesgos en nuestro análisis posterior.

Finalmente contabilizamos las "filas fantasma" de cada una de las tablas.

In [3]:
import pyspark.sql.functions as F

print("Iniciando limpieza de nulos y valores atípicos...")

# 1. Tratamiento de Ventas (stg_sales_typed)
# Llenamos los nulos con 0 y eliminamos devoluciones/errores (ventas negativas)
df_sales_clean = df_sales_typed \
    .fillna({"sales_qty": 0}) \
    .filter(F.col("sales_qty") >= 0)

# 2. Tratamiento de Precios (stg_prices_typed)
# Eliminamos filas sin precio (producto inactivo) y filtramos precios negativos o a 0
df_prices_clean = df_prices_typed \
    .dropna(subset=["sell_price"]) \
    .filter(F.col("sell_price") > 0)

# 3. Tratamiento de Calendario (stg_calendar_typed)
# El calendario suele venir limpio, pero eliminamos cualquier fecha nula por seguridad
df_calendar_clean = df_calendar_typed \
    .dropna(subset=["date"])

print("¡Limpieza completada!")


print("Calculando volumetría antes y después de la limpieza (esto puede tardar unos segundos)...")

# 1. Conteo de la tabla original (tipada)
sales_before = df_sales_typed.count()
prices_before = df_prices_typed.count()
calendar_before = df_calendar_typed.count()

# 2. Conteo de la tabla limpia
sales_after = df_sales_clean.count()
prices_after = df_prices_clean.count()
calendar_after = df_calendar_clean.count()

# 3. Cálculo de la diferencia (Filas fantasma o atípicas eliminadas)
sales_removed = sales_before - sales_after
prices_removed = prices_before - prices_after
calendar_removed = calendar_before - calendar_after

# 4. Reporte de auditoría
print("-" * 40)
print("AUDITORÍA DE LIMPIEZA DE DATOS M5")
print("-" * 40)
print(f"Ventas (stg_sales):")
print(f"  - Filas originales: {sales_before:,}")
print(f"  - Filas limpias:    {sales_after:,}")
print(f"  - Filas eliminadas (negativas): {sales_removed:,} ({((sales_removed/sales_before)*100):.2f}%)\n")

print(f"Precios (stg_prices):")
print(f"  - Filas originales: {prices_before:,}")
print(f"  - Filas limpias:    {prices_after:,}")
print(f"  - Filas eliminadas (sin precio/0): {prices_removed:,} ({((prices_removed/prices_before)*100):.2f}%)\n")

print(f"Calendario (stg_calendar):")
print(f"  - Filas originales: {calendar_before:,}")
print(f"  - Filas limpias:    {calendar_after:,}")
print(f"  - Filas eliminadas (nulas): {calendar_removed:,} ({((calendar_removed/calendar_before)*100):.2f}%)")
print("-" * 40)

Iniciando limpieza de nulos y valores atípicos...
¡Limpieza completada!
Calculando volumetría antes y después de la limpieza (esto puede tardar unos segundos)...
----------------------------------------
AUDITORÍA DE LIMPIEZA DE DATOS M5
----------------------------------------
Ventas (stg_sales):
  - Filas originales: 58,327,370
  - Filas limpias:    58,327,370
  - Filas eliminadas (negativas): 0 (0.00%)

Precios (stg_prices):
  - Filas originales: 6,841,121
  - Filas limpias:    6,841,121
  - Filas eliminadas (sin precio/0): 0 (0.00%)

Calendario (stg_calendar):
  - Filas originales: 1,969
  - Filas limpias:    1,969
  - Filas eliminadas (nulas): 0 (0.00%)
----------------------------------------


### 🔗 Unión de Tablas (Joins)

Vamos a consolidar la información uniendo nuestros DataFrames en dos pasos lógicos y secuenciales:
* **Paso 1:** Cruzamos `Ventas` con `Calendario` ➔ *Objetivo: Identificar en qué semana exacta ocurrió cada transacción.*
* **Paso 2:** Cruzamos el resultado anterior con `Precios` ➔ *Objetivo: Asignar el precio histórico correspondiente a esa misma semana.*

💡 **Estrategia de cruce:** Realizar estas uniones de forma escalonada nos ayuda a mantener un control claro sobre la granularidad de los datos (nivel semanal) y previene la duplicación accidental de registros.

In [4]:
import pyspark.sql.functions as F

print("Iniciando la construcción de la Tabla Maestra (Joins)...")

# 1. Primer Join: Ventas + Calendario
# Clave en común: La columna 'd' (ej. 'd_1', 'd_2')
# Usamos F.broadcast() porque el calendario es muy pequeño.
# Usamos un 'left' join para mantener todas nuestras ventas intactas.
df_sales_cal = df_sales_clean.join(
    F.broadcast(df_calendar_clean),
    on="d",
    how="left"
)

# 2. Segundo Join: Resultado Anterior + Precios
# Claves en común: Tienda, Producto y Semana (store_id, item_id, wm_yr_wk)
# Aquí no usamos broadcast porque la tabla de precios es mediana (casi 7M de filas).
df_master = df_sales_cal.join(
    df_prices_clean,
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left"
)

print("¡Tabla Maestra generada en memoria!")

# Mostramos el esquema resultante y unas filas de muestra para validar
df_master.printSchema()
df_master.show(5)

Iniciando la construcción de la Tabla Maestra (Joins)...
¡Tabla Maestra generada en memoria!
root
 |-- store_id: string (nullable = true)
 |-- item_id: string (nullable = true)
 |-- wm_yr_wk: integer (nullable = true)
 |-- d: string (nullable = true)
 |-- id: string (nullable = true)
 |-- dept_id: string (nullable = true)
 |-- cat_id: string (nullable = true)
 |-- state_id: string (nullable = true)
 |-- sales_qty: integer (nullable = false)
 |-- date: date (nullable = true)
 |-- weekday: string (nullable = true)
 |-- wday: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- event_name_1: string (nullable = true)
 |-- event_type_1: string (nullable = true)
 |-- event_name_2: string (nullable = true)
 |-- event_type_2: string (nullable = true)
 |-- snap_ca: integer (nullable = true)
 |-- snap_tx: integer (nullable = true)
 |-- snap_wi: integer (nullable = true)
 |-- sell_price: float (nullable = true)

+--------+-------------+--------

### ⚠️ Control de Volumetría: Prevención de *Fan-out*

El mayor riesgo al realizar cruces de tablas es sufrir una **Explosión Cartesiana** (o *Fan-out*). Para entender la gravedad de este problema en nuestro caso:
* **El origen:** Si por accidente la tabla de `Precios` tuviera dos precios distintos para el mismo producto en la misma semana, el `left join` duplicaría la fila de ventas para emparejarla con ambos.
* **El impacto:** Pasaríamos de tener 58 millones de filas a cientos de millones de forma silenciosa, arruinando por completo la precisión del modelo predictivo.

💡 **Validación rápida:** A continuación, ejecutaremos un conteo de filas sobre nuestro `df_master`. Si el cruce se ha realizado correctamente, la volumetría debe mantenerse idéntica a la de nuestra tabla base de ventas.

In [5]:
import pyspark.sql.functions as F

print("Ejecutando cruces masivos (JOINS) para TODO el dataset (58M+ filas)...")

# 1. Contamos las filas base completas (esto puede tardar un poco)
print("Contando filas base de ventas...")
sales_total_count = df_sales_clean.count()

# 2. Aplicamos la receta de Joins a todo el volumen
print("Cruzando ventas con el calendario...")
df_sales_cal = df_sales_clean.join(
    F.broadcast(df_calendar_clean), # Mantenemos el broadcast porque el calendario es muy pequeño
    on="d",
    how="left"
)

print("Cruzando resultado con los precios...")
df_master = df_sales_cal.join(
    df_prices_clean,
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left"
)

# 3. Contamos el resultado cruzado final
print("Contando filas tras los cruces (generando df_master)...")
master_total_count = df_master.count()

# 4. Auditoría a gran escala
print("-" * 40)
print("AUDITORÍA DE JOINS (DATASET COMPLETO)")
print("-" * 40)
print(f"Filas limpias originales: {sales_total_count:,}")
print(f"Filas tras el cruce:      {master_total_count:,}")
print("-" * 40)

if master_total_count == sales_total_count:
    print("✅ ¡Éxito absoluto! La lógica del JOIN escaló perfectamente sin multiplicar filas.")
else:
    print(f"❌ ALERTA: Diferencia detectada de {master_total_count - sales_total_count:,} filas. Revisa las claves de cruce.")

Ejecutando cruces masivos (JOINS) para TODO el dataset (58M+ filas)...
Contando filas base de ventas...
Cruzando ventas con el calendario...
Cruzando resultado con los precios...
Contando filas tras los cruces (generando df_master)...
----------------------------------------
AUDITORÍA DE JOINS (DATASET COMPLETO)
----------------------------------------
Filas limpias originales: 58,327,370
Filas tras el cruce:      58,327,370
----------------------------------------
✅ ¡Éxito absoluto! La lógica del JOIN escaló perfectamente sin multiplicar filas.



### 🛠️ Feature Engineering: Indicador de Fin de Semana

Nos falta añadir una de las variables con mayor poder predictivo para nuestro análisis: identificar si la transacción ocurrió en fin de semana. Para implementarlo:
* **Nueva variable:** Indicador de fin de semana (1/0).
* **Método:** Utilizaremos la función condicional `when` nativa de PySpark.
* **Aplicación:** Esta transformación se integrará directamente en nuestro `df_master_test`.

💡 **Impacto en el modelo:** Los patrones de consumo y ventas suelen cambiar drásticamente entre días laborables y fines de semana. Aislar este comportamiento en una nueva característica (*feature*) ayudará al algoritmo predictivo a capturar esta estacionalidad a corto plazo.

### ⏳ Feature Engineering: Variables Históricas

Para dotar a nuestro modelo de contexto temporal, crearemos nuevas columnas que miren al pasado utilizando dos técnicas fundamentales en series temporales:
* **Lags (Rezagos):** Representan saltos exactos en el tiempo. Por ejemplo, un `lag_7` copia exactamente lo que se vendió hace una semana. En el sector *retail*, los periodos de 7, 14 y 28 días son la clave para capturar la estacionalidad.
* **Rolling Windows (Medias Móviles):** Permiten suavizar el "ruido" de la serie (como una caída puntual de ventas porque llovió un día), calculando el promedio sobre una ventana de tiempo específica.

💡 **Prevención de Data Leakage (Fuga de Datos):** Tenemos que ser muy estrictos al programar esto para no hacer "trampas". Si al calcular la media móvil incluimos accidentalmente las ventas de *hoy*, le estaríamos dando al modelo la respuesta antes de que intente predecirla. Por ello, definiremos en el código que la ventana mire estrictamente hacia los días pasados (del `-7` al `-1`).

In [6]:
from pyspark.sql.window import Window
import pyspark.sql.functions as F

print("Calculando Lags y Medias Móviles para TODO el dataset...")

# 1. Ventana temporal global particionada por el ID único de producto-tienda
window_spec = Window.partitionBy("id").orderBy("date")
window_rolling_7 = window_spec.rowsBetween(-7, -1)
window_rolling_28 = window_spec.rowsBetween(-28, -1)

# 2. Aplicamos las transformaciones sobre df_master completo
df_lags_full = df_master \
    .withColumn("lag_7", F.lag("sales_qty", 7).over(window_spec)) \
    .withColumn("lag_14", F.lag("sales_qty", 14).over(window_spec)) \
    .withColumn("lag_28", F.lag("sales_qty", 28).over(window_spec)) \
    .withColumn("rolling_mean_7", F.avg("sales_qty").over(window_rolling_7)) \
    .withColumn("rolling_mean_28", F.avg("sales_qty").over(window_rolling_28))

print("¡Variables históricas globales calculadas!")
# 3. Eliminamos el periodo de calentamiento inicial (los primeros 28 días) 
# usando subset para no perder los datos del calendario
df_final_full = df_lags_full.dropna(subset=["lag_28", "sell_price"])

print("¡Dataset limpio y preparado para producción!")

Calculando Lags y Medias Móviles para TODO el dataset...
¡Variables históricas globales calculadas!
¡Dataset limpio y preparado para producción!


In [7]:
df_final_full.show(5)

+--------+-----------+--------+----+--------------------+-------+------+--------+---------+----------+---------+----+-----+----+------------+------------+------------+------------+-------+-------+-------+----------+-----+------+------+-------------------+-------------------+
|store_id|    item_id|wm_yr_wk|   d|                  id|dept_id|cat_id|state_id|sales_qty|      date|  weekday|wday|month|year|event_name_1|event_type_1|event_name_2|event_type_2|snap_ca|snap_tx|snap_wi|sell_price|lag_7|lag_14|lag_28|     rolling_mean_7|    rolling_mean_28|
+--------+-----------+--------+----+--------------------+-------+------+--------+---------+----------+---------+----+-----+----+------------+------------+------------+------------+-------+-------+-------+----------+-----+------+------+-------------------+-------------------+
|    CA_3|FOODS_1_002|   11105|d_29|FOODS_1_002_CA_3_...|FOODS_1| FOODS|      CA|        1|2011-02-26| Saturday|   1|    2|2011|        NULL|        NULL|        NULL|     

In [8]:
import os

# 1. Forzamos la creación de una ruta absoluta estilo Windows/Spark
directorio_actual = os.getcwd()
output_path = os.path.join(directorio_actual, "data", "processed", "abt_master_full.parquet")
output_path = output_path.replace("\\", "/") # Cambiamos las barras para que Hadoop no se queje

print(f"Exportando el dataset completo a Parquet...")
print(f"Ruta absoluta destino: {output_path}")

# 2. Configuración antimanchas de Hadoop
spark.conf.set("mapreduce.fileoutputcommitter.marksuccessfuljobs", "false")

# 3. Escritura masiva
df_final_full.write \
    .mode("overwrite") \
    .partitionBy("store_id") \
    .parquet(output_path)

print("✅ ¡Exportación masiva completada con éxito!")

Exportando el dataset completo a Parquet...
Ruta absoluta destino: d:/Usuarios/Escritorio/M5/m5-forecasting-project/notebooks/data/processed/abt_master_full.parquet
✅ ¡Exportación masiva completada con éxito!
